Filename: notebook_02.ipynb <br>
Description: Mall_Customers dataset 

---

0. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN
)
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11

---

1. Load Dataset

In [ ]:
df = pd.read_csv('mall_customers.csv')

# Display first 5 rows
display(df.head())

---

2. Data Understanding

In [ ]:
# Dataset dimensions
print('Shape:', df.shape)

# Dataset information
df.info()

# Descriptive statistics
display(df.describe())

# Check missing values
print(df.isnull().sum())

# Check duplicated records
print('Duplicates:', df.duplicated().sum())

# Check column names
print(df.columns)

---

3. Exploratory Data Analysis (EDA)

<small>3.1 Distribution of Numerical Features<small>

In [ ]:
num_cols = [
    'Age',
    'Annual Income (k$)',
    'Spending Score (1-100)'
]

df[num_cols].hist(
    figsize=(12, 4),
    bins=15,
    edgecolor='black'
)

plt.tight_layout()
plt.show()

<small>3.2 Boxplot – Outlier Detection<small>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for col, ax in zip(num_cols, axes):
    sns.boxplot(
        data=df,
        y=col,
        ax=ax
    )
    ax.set_title(col)
plt.tight_layout()
plt.show()

<small>3.3 Gender Distribution</small>

In [ ]:
sns.countplot(
    data=df,
    x='Gender'
)

plt.title('Customer Gender Distribution')
plt.show()

<small>3.4 Correlation Matrix</small>

In [ ]:
plt.figure(figsize=(7, 5))

sns.heatmap(
    df[num_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Matrix')
plt.show()

<small>3.5 Scatter Plot: Annual Income vs Spending Score</small>

In [ ]:
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Gender',
    s=70
)

plt.title('Annual Income vs Spending Score')
plt.show()

---

4. Feature Selection

<small>Model A: 2 Features</small>

In [ ]:
features_A = [
    'Annual Income (k$)',
    'Spending Score (1-100)'
]

X_A = df[features_A].copy()

print(X_A.head())

<small>Model B: 3 Features</small>

In [ ]:
features_B = [
    'Age',
    'Annual Income (k$)',
    'Spending Score (1-100)'
]

X_B = df[features_B].copy()

print(X_B.head())

---

5. Feature Scaling

<small>5.1 StandardScaler</small>

In [ ]:
scaler_std = StandardScaler()

X_scaled_std = scaler_std.fit_transform(X_A)

print('Mean:', X_scaled_std.mean(axis=0))
print('Std:', X_scaled_std.std(axis=0))

<small>5.2 MinMaxScaler</small>

In [ ]:
scaler_mm = MinMaxScaler()

X_scaled_mm = scaler_mm.fit_transform(X_A)

print('Min:', X_scaled_mm.min(axis=0))
print('Max:', X_scaled_mm.max(axis=0))

<small>5.3 Compare Before and After Scaling</small>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(
    X_A.iloc[:, 0],
    X_A.iloc[:, 1]
)
axes[0].set_title('Original Data')

axes[1].scatter(
    X_scaled_std[:, 0],
    X_scaled_std[:, 1]
)
axes[1].set_title('StandardScaler')

axes[2].scatter(
    X_scaled_mm[:, 0],
    X_scaled_mm[:, 1]
)
axes[2].set_title('MinMaxScaler')

plt.tight_layout()
plt.show()

---

6. K-Means Clustering

<small>6.1 Elbow Method</small>

In [ ]:
wcss = []

for k in range(1, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled_std)

    wcss.append(kmeans.inertia_)


plt.figure(figsize=(8, 5))

plt.plot(
    range(1, 11),
    wcss,
    marker='o'
)

plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS')
plt.title('Elbow Method')

plt.xticks(range(1, 11))
plt.show()

<small>6.2 Silhouette Score</small>

In [ ]:
silhouette_scores = []

for k in range(2, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled_std)

    score = silhouette_score(
        X_scaled_std,
        labels
    )

    silhouette_scores.append(score)


plt.figure(figsize=(8, 5))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker='o'
)

plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')

plt.xticks(range(2, 11))
plt.show()

<small>6.3 Apply K-Means</small>

In [ ]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

df['KMeans_Cluster'] = kmeans.fit_predict(
    X_scaled_std
)

print(df['KMeans_Cluster'].value_counts().sort_index())

<small>6.4 Visualize K-Means Clusters<small>

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    x=X_A.iloc[:, 0],
    y=X_A.iloc[:, 1],
    hue=df['KMeans_Cluster'],
    palette='tab10',
    s=80
)

# Plot Centroids (inverse scaling)
centroids = scaler_std.inverse_transform(
    kmeans.cluster_centers_
)

plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    marker='X',
    s=250,
    color='red',
    edgecolor='black',
    label='Centroids'
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('K-Means Customer Segmentation')

plt.legend()
plt.show()

<small>6.5 Compare K-Means with Different Features</small>

In [ ]:
# Model B: 3 Features
scaler_B = StandardScaler()

X_B_scaled = scaler_B.fit_transform(X_B)

kmeans_B = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

labels_B = kmeans_B.fit_predict(X_B_scaled)

print(
    'Model A Silhouette:',
    silhouette_score(X_scaled_std, df['KMeans_Cluster'])
)

print(
    'Model B Silhouette:',
    silhouette_score(X_B_scaled, labels_B)
)

---

7. Hierarchical Clustering

<small>7.1 Dendrogram

In [ ]:
Z = linkage(
    X_scaled_std,
    method='ward'
)

plt.figure(figsize=(14, 6))

dendrogram(
    Z,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90,
    leaf_font_size=10
)

plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Cluster')
plt.ylabel('Distance')

plt.show()

<small>7.2 Compare Linkage Methods

In [ ]:
linkage_methods = [
    'single',
    'complete',
    'average',
    'ward'
]

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10)
)

for method, ax in zip(linkage_methods, axes.flatten()):

    Z = linkage(
        X_scaled_std,
        method=method
    )

    dendrogram(
        Z,
        truncate_mode='lastp',
        p=25,
        ax=ax
    )

    ax.set_title(f'{method.capitalize()} Linkage')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Distance')

plt.tight_layout()
plt.show()

<small>7.3 Apply Agglomerative Clustering

In [ ]:
hc = AgglomerativeClustering(
    n_clusters=5,
    linkage='ward'
)

df['HC_Cluster'] = hc.fit_predict(
    X_scaled_std
)

print(df['HC_Cluster'].value_counts().sort_index())

<small>7.4 Visualize Hierarchical Clusters</small>

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    x=X_A.iloc[:, 0],
    y=X_A.iloc[:, 1],
    hue=df['HC_Cluster'],
    palette='tab10',
    s=80
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Hierarchical Clustering')

plt.show()

---

8. DBSCAN Clustering

<small>8.1 K-Distance Graph</small>

In [ ]:
min_samples = 5

nearest_neighbors = NearestNeighbors(
    n_neighbors=min_samples
)

nearest_neighbors.fit(X_scaled_std)

distances, indices = nearest_neighbors.kneighbors(
    X_scaled_std
)

# Distance to the k-th nearest neighbor
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(9, 5))

plt.plot(k_distances)

plt.xlabel('Points Sorted by Distance')
plt.ylabel(f'{min_samples}-NN Distance')
plt.title('K-Distance Graph')

plt.show()

<small>8.2 Apply DBSCAN</small>

In [ ]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=5
)

df['DBSCAN_Cluster'] = dbscan.fit_predict(
    X_scaled_std
)

print(
    df['DBSCAN_Cluster'].value_counts().sort_index()
)

<small>8.3 Visualize DBSCAN Clusters</small>

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    x=X_A.iloc[:, 0],
    y=X_A.iloc[:, 1],
    hue=df['DBSCAN_Cluster'],
    palette='tab10',
    s=80
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('DBSCAN Clustering')

plt.show()

<small>8.4 Identify Core, Border and Noise Points</small>

In [ ]:
# Identify Core Points
core_indices = dbscan.core_sample_indices_

core_mask = np.zeros(len(X_scaled_std), dtype=bool)
core_mask[core_indices] = True

# Identify Noise Points
noise_mask = df['DBSCAN_Cluster'].values == -1

# Identify Border Points
border_mask = (~core_mask) & (~noise_mask)

print('Core Points:', core_mask.sum())
print('Border Points:', border_mask.sum())
print('Noise Points:', noise_mask.sum())

<small>8.5 Visualize Core, Border and Noise</small>

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    X_A.iloc[core_mask, 0],
    X_A.iloc[core_mask, 1],
    c='blue',
    s=60,
    label='Core Points'
)

plt.scatter(
    X_A.iloc[border_mask, 0],
    X_A.iloc[border_mask, 1],
    c='orange',
    s=60,
    label='Border Points'
)

plt.scatter(
    X_A.iloc[noise_mask, 0],
    X_A.iloc[noise_mask, 1],
    c='red',
    marker='x',
    s=80,
    label='Noise Points'
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('DBSCAN: Core, Border and Noise Points')

plt.legend()
plt.show()

<small>8.6 Experiment with Different eps Values</small>

In [ ]:
eps_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 9)
)

for eps, ax in zip(eps_values, axes.flatten()):

    model = DBSCAN(
        eps=eps,
        min_samples=5
    )

    labels = model.fit_predict(X_scaled_std)

    ax.scatter(
        X_A.iloc[:, 0],
        X_A.iloc[:, 1],
        c=labels,
        cmap='tab10',
        s=40
    )

    n_clusters = len(set(labels) - {-1})
    n_noise = np.sum(labels == -1)

    ax.set_title(
        f'eps={eps} | Clusters={n_clusters} | Noise={n_noise}'
    )

    ax.set_xlabel('Annual Income')
    ax.set_ylabel('Spending Score')

plt.tight_layout()
plt.show()

---

9. Clustering Evaluation

<small>9.1 Calculate Evaluation Metrics</small>

In [ ]:
def evaluate_clustering(X, labels):

    labels = np.asarray(labels)

    # Exclude Noise Points (-1)
    mask = labels != -1

    X_eval = X[mask]
    labels_eval = labels[mask]

    n_clusters = len(np.unique(labels_eval))
    n_noise = np.sum(~mask)

    # Metrics require at least 2 clusters
    if n_clusters < 2 or n_clusters >= len(labels_eval):

        return {
            'Clusters': n_clusters,
            'Noise': n_noise,
            'Silhouette': np.nan,
            'Davies-Bouldin': np.nan,
            'Calinski-Harabasz': np.nan
        }

    return {
        'Clusters': n_clusters,
        'Noise': n_noise,

        'Silhouette': silhouette_score(
            X_eval,
            labels_eval
        ),

        'Davies-Bouldin': davies_bouldin_score(
            X_eval,
            labels_eval
        ),

        'Calinski-Harabasz': calinski_harabasz_score(
            X_eval,
            labels_eval
        )
    }

<small>9.2 Compare Algorithms</small>

In [ ]:
results = []

models = {
    'K-Means': df['KMeans_Cluster'].values,
    'Hierarchical': df['HC_Cluster'].values,
    'DBSCAN': df['DBSCAN_Cluster'].values
}

for name, labels in models.items():

    scores = evaluate_clustering(
        X_scaled_std,
        labels
    )

    scores['Algorithm'] = name

    results.append(scores)


evaluation_df = pd.DataFrame(results)

evaluation_df = evaluation_df.set_index('Algorithm')

display(evaluation_df.round(4))

#Silhouette Score: ค่ายิ่งสูงยิ่งดี 
#Davies-Bouldin Index: ค่ายิ่งต่ำยิ่งดี 
#Calinski-Harabasz Index: ค่ายิ่งสูงยิ่งดี 

<small>9.3 Visualize Silhouette Comparison</small>

In [ ]:
evaluation_df['Silhouette'].plot(
    kind='bar',
    figsize=(8, 5)
)

plt.ylabel('Silhouette Score')
plt.title('Clustering Algorithm Comparison')
plt.xticks(rotation=0)

plt.show()

---

10. Interpreting Clustering Results

<small>10.1 Cluster Size</small>

In [ ]:
cluster_counts = df['KMeans_Cluster'].value_counts().sort_index()

print(cluster_counts)

cluster_counts.plot(
    kind='bar',
    figsize=(8, 5)
)

plt.xlabel('Cluster')
plt.ylabel('Number of Customers')
plt.title('Number of Customers in Each Cluster')

plt.show()

<small>10.2 Cluster Profile</small>

In [ ]:
cluster_profile = df.groupby('KMeans_Cluster')[
    [
        'Age',
        'Annual Income (k$)',
        'Spending Score (1-100)'
    ]
].mean().round(2)

display(cluster_profile)

<small>10.3 Cluster Distribution by Gender</small>

In [ ]:
gender_profile = pd.crosstab(
    df['KMeans_Cluster'],
    df['Gender'],
    normalize='index'
) * 100

display(gender_profile.round(2))

<small>10.4 Visualize Cluster Profiles</small>

In [ ]:
cluster_profile.plot(
    kind='bar',
    subplots=True,
    layout=(1, 3),
    figsize=(15, 5),
    legend=False
)

plt.suptitle('Customer Profile by Cluster')
plt.tight_layout()
plt.show()

<small>10.5 Compare Mean Values Across Clusters</small>

In [ ]:
cluster_profile.T.plot(
    kind='bar',
    figsize=(11, 5)
)

plt.ylabel('Average Value')
plt.title('Feature Comparison Across Clusters')
plt.xticks(rotation=0)

plt.legend(title='Cluster')
plt.show()

---

11. Final Customer Segmentation

In [ ]:
profile = cluster_profile.copy()

income_median = df['Annual Income (k$)'].median()
spending_median = df['Spending Score (1-100)'].median()

def assign_segment(row):

    income = row['Annual Income (k$)']
    spending = row['Spending Score (1-100)']

    if income >= income_median and spending >= spending_median:
        return 'High Income - High Spending'

    elif income >= income_median and spending < spending_median:
        return 'High Income - Low Spending'

    elif income < income_median and spending >= spending_median:
        return 'Low Income - High Spending'

    else:
        return 'Low Income - Low Spending'


profile['Customer Segment'] = profile.apply(
    assign_segment,
    axis=1
)

display(profile)

<small>11.1 Create Final Summary Table</small>

In [ ]:
summary = df.groupby('KMeans_Cluster').agg(
    Number_of_Customers=('CustomerID', 'count'),
    Average_Age=('Age', 'mean'),
    Average_Income=('Annual Income (k$)', 'mean'),
    Average_Spending=('Spending Score (1-100)', 'mean')
).round(2)

summary['Customer_Segment'] = profile['Customer Segment']

display(summary)

---

12. Final Visualization: Customer Segmentation

In [ ]:
plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='KMeans_Cluster',
    palette='tab10',
    s=100
)

centroids = scaler_std.inverse_transform(
    kmeans.cluster_centers_
)

plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    marker='X',
    s=300,
    color='red',
    edgecolor='black',
    label='Centroids'
)

plt.title(
    'Mall Customer Segmentation',
    fontsize=16
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')

plt.legend()
plt.show()

---

13. Export Results

In [ ]:
df.to_csv(
    'Mall_Customer_Clustering_Results.csv',
    index=False
)

print('Export completed!')

---